# Mixture of Experts（MoE）从原理到 PyTorch 实现

> 本笔记目标：从普通 FFN 出发，逐步实现 **SwiGLU 专家、Top-k Router、稀疏分发、负载均衡损失、共享专家**，最后对照 `agent_policy` 中的真实 MoE 代码与配置。

MoE 的核心不是“把多个模型平均”，而是：**对每个 token，只激活少数几个专家网络**。这样可以显著增加模型参数容量，而单个 token 的计算量只随激活专家数 $K$ 增长，而不是随总专家数 $E$ 增长。

本笔记覆盖：

1. Dense FFN、SwiGLU 与稀疏 MoE 的关系；
2. Router、Top-k、门控加权的数学定义；
3. 每一步张量形状与可运行的纯 PyTorch 实现；
4. 专家塌缩、负载均衡、capacity、token dropping；
5. 训练、推理、分布式 Expert Parallel 的工程差异；
6. `agent_policy` 的接入位置、配置参数、实际实现及注意事项。

---

## 1. 一句话直觉

普通 Transformer 的每一层都让所有 token 经过同一个 FFN；MoE 则准备 $E$ 个 FFN，由 Router 给每个 token 选择 Top-$K$ 个专家：

$$
y_t = \sum_{e \in \operatorname{TopK}(p_t)} \tilde p_{t,e}\,\operatorname{Expert}_e(x_t) + \operatorname{SharedExpert}(x_t).
$$

- $x_t\in\mathbb R^H$：第 $t$ 个 token；
- $p_t=\operatorname{softmax}(W_rx_t)\in\mathbb R^E$：对全部专家的路由概率；
- $\tilde p_{t,e}$：只保留 Top-$K$ 后重新归一化的权重；
- Routed experts：只处理被分配给自己的 token；
- Shared expert：可选，所有 token 都经过它，用于承载通用知识。

## 2. Dense、Ensemble 与 Sparse MoE 的区别

| 结构 | 每个 token 激活多少网络 | 输出如何组合 | 主要目的 |
|---|---:|---|---|
| Dense FFN | 1 个共享 FFN | 直接输出 | 标准 Transformer |
| Ensemble | 通常激活全部独立模型 | 模型级平均/投票 | 提升鲁棒性 |
| Dense MoE | $E$ 个专家全算 | 概率加权和 | 易实现但计算昂贵 |
| Sparse MoE | 仅 Top-$K$ 个专家 | Top-$K$ 加权和 | 扩大参数容量而控制 FLOPs |

若单个专家中间维度为 $I$，忽略 bias，则一个 SwiGLU 专家的参数量约为 $3HI$：两个上投影与一个下投影。$E$ 个 routed experts 的参数量约为 $3EHI$，但理想稀疏执行时每个 token 只计算约 $3KHI$。注意：**参数稀疏不自动等于运行更快**；若代码仍把所有专家都计算一遍，就只有语义稀疏，没有计算稀疏。

### 本笔记统一形状符号

- `B`：batch size；
- `N`：每个样本的 token 数；
- `T = B * N`：展平后的 token 总数；
- `H`：token hidden dimension；
- `I`：专家中间维度；
- `E`：routed expert 总数；
- `K`：每个 token 激活的专家数。

In [1]:
# 只依赖 PyTorch；固定随机种子保证每次运行结果可复现。
import math
from dataclasses import dataclass

import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(7)
torch.set_printoptions(precision=4, sci_mode=False)

# 教学用小尺寸：x 的形状为 [B, N, H] = [2, 5, 16]。
B, N, H = 2, 5, 16
I, E, K = 32, 4, 2
x = torch.randn(B, N, H)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("PyTorch:", torch.__version__)
print("可用设备:", device)
print("x.shape:", x.shape, "；展平后 token 数 T =", B * N)

PyTorch: 2.0.1+cu118
可用设备: cuda
x.shape: torch.Size([2, 5, 16]) ；展平后 token 数 T = 10


## 3. 从普通 FFN 到 SwiGLU 专家

经典 Transformer FFN 常写为：

$$\operatorname{FFN}(x)=W_{down}\,\sigma(W_{up}x).$$

SwiGLU 使用一条 gate 分支与一条 value 分支：

$$\operatorname{SwiGLU}(x)=W_{down}\big(\operatorname{SiLU}(W_gx)\odot W_ux\big).$$

对输入 `[..., H]`：`gate_proj` 与 `up_proj` 均产生 `[..., I]`，逐元素乘法后由 `down_proj` 投回 `[..., H]`。每个专家输入输出维度相同，因此能被 Router 加权组合，也能直接替换普通 FFN。

In [2]:
class SwiGLUExpert(nn.Module):
    '''一个独立专家；支持任意前导维度，只要求最后一维是 hidden_dim。'''

    def __init__(self, hidden_dim: int, intermediate_size: int):
        super().__init__()
        self.gate_proj = nn.Linear(hidden_dim, intermediate_size, bias=False)
        self.up_proj = nn.Linear(hidden_dim, intermediate_size, bias=False)
        self.down_proj = nn.Linear(intermediate_size, hidden_dim, bias=False)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: [..., H]
        gate = F.silu(self.gate_proj(x))  # [..., I]
        value = self.up_proj(x)           # [..., I]
        hidden = gate * value             # [..., I]，逐元素门控
        return self.down_proj(hidden)      # [..., H]


expert = SwiGLUExpert(H, I)
expert_y = expert(x)
assert expert_y.shape == x.shape
parameter_count = sum(p.numel() for p in expert.parameters())
print("expert(x).shape =", expert_y.shape)
print(f"参数量 = {parameter_count} = 3 × H × I = {3 * H * I}")

expert(x).shape = torch.Size([2, 5, 16])
参数量 = 1536 = 3 × H × I = 1536


## 4. Router：为每个 token 选择专家

最简单的线性 Router 使用可学习矩阵 $W_r\in\mathbb R^{E\times H}$：

1. `x [B,N,H] → x_flat [T,H]`；
2. `logits = x_flat @ W_r.T → [T,E]`；
3. `scores = softmax(logits) → [T,E]`；
4. 对最后一维取 Top-$K$，得到 `topk_idx [T,K]` 和 `topk_weight [T,K]`；
5. 把被选中的 $K$ 个概率再次归一化，使每个 token 的组合权重之和为 1。

Top-k 的索引选择是离散操作，未选专家通常得不到该 token 的专家/路由梯度；被选中权重仍是可微的。实际大模型还常用 router logits 的 FP32 计算、router jitter/noise、z-loss 等稳定技巧。

In [3]:
@dataclass
class RouterOutput:
    topk_idx: torch.Tensor       # [T, K]，整数专家编号
    topk_weight: torch.Tensor    # [T, K]，Top-K 内归一化权重
    scores: torch.Tensor         # [T, E]，全部专家的 softmax 概率
    logits: torch.Tensor         # [T, E]，未归一化路由分数


class TopKRouter(nn.Module):
    def __init__(self, hidden_dim: int, num_experts: int, top_k: int):
        super().__init__()
        if not 1 <= top_k <= num_experts:
            raise ValueError("必须满足 1 <= top_k <= num_experts")
        self.num_experts = num_experts
        self.top_k = top_k
        self.weight = nn.Parameter(torch.empty(num_experts, hidden_dim))  # [E,H]
        nn.init.kaiming_uniform_(self.weight, a=math.sqrt(5))

    def forward(self, x: torch.Tensor) -> RouterOutput:
        # x: [B,N,H]；也兼容任意 [...,H]。
        x_flat = x.reshape(-1, x.shape[-1])                  # [T,H]
        logits = F.linear(x_flat.float(), self.weight.float())  # [T,E]，FP32 更稳
        scores = F.softmax(logits, dim=-1)                   # [T,E]
        topk_weight, topk_idx = torch.topk(
            scores, k=self.top_k, dim=-1, sorted=False
        )                                                    # 均为 [T,K]
        topk_weight = topk_weight / topk_weight.sum(dim=-1, keepdim=True).clamp_min(1e-20)
        return RouterOutput(topk_idx, topk_weight.to(x.dtype), scores, logits)


router = TopKRouter(H, E, K)
r = router(x)
print("logits / scores:", r.logits.shape, r.scores.shape)
print("topk_idx / topk_weight:", r.topk_idx.shape, r.topk_weight.shape)
print("前 3 个 token 的专家编号:\n", r.topk_idx[:3])
print("前 3 个 token 的权重:\n", r.topk_weight[:3])
print("每个 token 的 Top-K 权重和:\n", r.topk_weight.sum(-1))
assert torch.allclose(r.topk_weight.sum(-1), torch.ones(B * N), atol=1e-6)

logits / scores: torch.Size([10, 4]) torch.Size([10, 4])
topk_idx / topk_weight: torch.Size([10, 2]) torch.Size([10, 2])


前 3 个 token 的专家编号:
 tensor([[0, 1],
        [3, 2],
        [3, 1]])
前 3 个 token 的权重:
 

tensor([[0.5095, 0.4905],
        [0.5831, 0.4169],
        [0.5294, 0.4706]], grad_fn=<SliceBackward0>)
每个 token 的 Top-K 权重和:
 tensor([1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000,
        1.0000], grad_fn=<SumBackward1>)


## 5. 稀疏分发（dispatch）与合并（combine）

路由之后要解决两个问题：

- **dispatch**：把 token 发送到对应专家；
- **combine**：按路由权重将多个专家输出加权求和。

教学实现采用“复制 $K$ 份 token，再按专家 mask 取子集”：

```text
x_flat [T,H]
  └─ repeat_interleave(K) → x_repeated [T*K,H]
topk_idx [T,K]
  └─ flatten              → flat_idx   [T*K]
每个专家 e 仅处理 x_repeated[flat_idx == e]
  └─ 拼回 [T,K,H]，乘 topk_weight [T,K,1]，沿 K 求和 → [T,H]
```

这种写法直观且确实只调用被选中的专家子集，适合学习和中小规模实验；生产级实现通常会按专家排序 token、做 capacity 管理，并使用 fused kernel 或跨设备 all-to-all。

In [4]:
class SparseMoE(nn.Module):
    '''Top-k routed experts + 可选 shared expert 的最小完整实现。'''

    def __init__(self, hidden_dim, intermediate_size, num_experts, top_k,
                 shared_intermediate_size=0):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.num_experts = num_experts
        self.top_k = top_k
        self.router = TopKRouter(hidden_dim, num_experts, top_k)
        self.experts = nn.ModuleList([
            SwiGLUExpert(hidden_dim, intermediate_size)
            for _ in range(num_experts)
        ])
        self.shared_expert = (
            SwiGLUExpert(hidden_dim, shared_intermediate_size)
            if shared_intermediate_size > 0 else None
        )

    def forward(self, x):
        # x: [B,N,H]；orig_shape 用来在末尾恢复批次与 token 维。
        orig_shape = x.shape
        x_flat = x.reshape(-1, self.hidden_dim)               # [T,H]
        route = self.router(x)                                # idx/weight: [T,K]

        # 每个 token 复制 K 次；第 t 个 token 的 K 个副本与其 K 个路由位置对齐。
        x_repeated = x_flat.repeat_interleave(self.top_k, 0)  # [T*K,H]
        flat_idx = route.topk_idx.reshape(-1)                 # [T*K]
        routed = torch.empty_like(x_repeated)                 # [T*K,H]

        expert_token_counts = []
        for expert_id, expert in enumerate(self.experts):
            mask = flat_idx == expert_id                      # [T*K] bool
            expert_token_counts.append(int(mask.sum()))
            if mask.any():
                routed[mask] = expert(x_repeated[mask])       # [T_e,H]

        routed = routed.reshape(-1, self.top_k, self.hidden_dim)  # [T,K,H]
        weight = route.topk_weight.unsqueeze(-1)                  # [T,K,1]
        y_flat = (routed * weight).sum(dim=1)                     # [T,H]
        y = y_flat.reshape(orig_shape)                            # [B,N,H]

        # shared expert 始终处理所有 token，其结果与 routed 输出相加。
        if self.shared_expert is not None:
            y = y + self.shared_expert(x)                         # [B,N,H]

        stats = {"route": route, "expert_token_counts": expert_token_counts}
        return y, stats


moe = SparseMoE(H, I, E, K, shared_intermediate_size=I)
y, stats = moe(x)
print("输入/输出形状:", x.shape, y.shape)
print("各专家收到的 token 副本数:", stats["expert_token_counts"])
print("总分配数:", sum(stats["expert_token_counts"]), "= T*K =", B*N*K)
assert y.shape == x.shape
assert sum(stats["expert_token_counts"]) == B * N * K

输入/输出形状: torch.Size([2, 5, 16]) torch.Size([2, 5, 16])
各专家收到的 token 副本数: [6, 6, 6, 2]
总分配数: 20 = T*K = 20


## 6. 为什么需要负载均衡损失

Router 很容易发生 **expert collapse**：大量 token 都选择少数专家，其他专家几乎没有样本与梯度；热门专家还会超过容量、造成通信与计算瓶颈。

一个常见的 Switch Transformer 风格辅助项为：

$$
L_{bal}=\alpha E\sum_{i=1}^{E}P_i f_i,
$$

其中：

- $P_i=\frac1T\sum_t p_{t,i}$：Router 对专家 $i$ 的平均概率，可微；
- $f_i=\frac1{TK}\sum_{t,k}\mathbb 1[e_{t,k}=i]$：实际 Top-$K$ 分配频率，选择本身不可微；
- 均匀路由时 $P_i=f_i=1/E$，未乘 `alpha` 的损失接近 1；越集中通常越大。

这里 $f_i$ 把一个 token 的 $K$ 次分配分别计数，因此分母隐含为 $TK$。不同论文对 Top-2、序列级/批次级统计、是否先乘权重的定义略有不同，复现时必须以源码为准。

In [5]:
def load_balance_loss(scores, topk_idx, alpha=1e-2):
    '''
    scores:   [T,E]，softmax 概率；保留梯度。
    topk_idx: [T,K]，离散专家编号。
    '''
    num_experts = scores.shape[-1]
    # [T,K,E]：每次 Top-K 分配转 one-hot；随后对 T 与 K 一起求平均。
    assignment = F.one_hot(topk_idx, num_classes=num_experts).float()
    f_i = assignment.mean(dim=(0, 1))  # [E]，实际负载占比，sum=1
    P_i = scores.mean(dim=0)            # [E]，平均路由概率，sum=1
    loss = alpha * num_experts * torch.sum(P_i * f_i)  # scalar
    return loss, P_i, f_i


r = stats["route"]
balance_loss, P_i, f_i = load_balance_loss(r.scores, r.topk_idx, alpha=0.01)
print("P_i（平均概率）:", P_i.detach())
print("f_i（实际负载）:", f_i)
print("sum(P_i), sum(f_i):", P_i.sum().item(), f_i.sum().item())
print("加权后的 balance loss:", balance_loss.item())

# 验证梯度确实能回到 Router 参数。
moe.zero_grad(set_to_none=True)
task_loss = y.square().mean()
balance_loss, _, _ = load_balance_loss(
    stats["route"].scores, stats["route"].topk_idx, alpha=0.01
)
total_loss = task_loss + balance_loss
total_loss.backward()
print("task loss / balance loss:", task_loss.item(), balance_loss.item())
print("router 梯度范数:", moe.router.weight.grad.norm().item())
assert moe.router.weight.grad is not None

P_i（平均概率）: tensor([0.2769, 0.2509, 0.2733, 0.1989])
f_i（实际负载）: tensor([0.3000, 0.3000, 0.3000, 0.1000])
sum(P_i), sum(f_i): 1.0 1.0
加权后的 balance loss: 0.010408780537545681


task loss / balance loss: 0.02087896689772606 0.010408780537545681


router 梯度范数: 0.003310735570266843


## 7. 一个可训练的最小 MoE 回归示例

下面构造异质数据：四类输入分别对应四种不同目标变换。它不是为了证明 Router 必然学出人类可解释的专家，而是演示完整训练闭环：`forward → task loss + balance loss → backward → optimizer.step`，并持续观察专家负载。

训练时通常写成：

$$L_{total}=L_{task}+\lambda_{bal}L_{bal}+\lambda_zL_z+\cdots$$

建议只在一个位置施加权重：要么 `load_balance_loss` 返回未加权项，由 Task 乘系数；要么函数内部乘系数，Task 直接相加，不要重复乘。

In [6]:
torch.manual_seed(11)
train_moe = SparseMoE(
    hidden_dim=8, intermediate_size=16, num_experts=4, top_k=1,
    shared_intermediate_size=8,
)
head = nn.Linear(8, 1)
optimizer = torch.optim.AdamW(
    list(train_moe.parameters()) + list(head.parameters()), lr=3e-3
)

for step in range(61):
    # [B,N,H] = [32,6,8]；cluster 决定样本属于哪种合成模式。
    cluster = torch.randint(0, 4, (32, 6))                    # [32,6]
    features = torch.randn(32, 6, 8)                          # [32,6,8]
    features[..., 0] += (cluster.float() - 1.5) * 2.0
    # 四种模式拥有不同的线性目标；target: [32,6,1]。
    slopes = torch.tensor([1.0, -1.0, 0.5, -0.5])
    target = slopes[cluster].unsqueeze(-1) * features[..., :1]

    hidden, info = train_moe(features)                         # [32,6,8]
    prediction = head(hidden)                                 # [32,6,1]
    regression_loss = F.mse_loss(prediction, target)           # scalar
    bal_loss, _, load = load_balance_loss(
        info["route"].scores, info["route"].topk_idx, alpha=0.02
    )
    loss = regression_loss + bal_loss

    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    torch.nn.utils.clip_grad_norm_(
        list(train_moe.parameters()) + list(head.parameters()), max_norm=1.0
    )
    optimizer.step()

    if step % 15 == 0:
        print(
            f"step={step:02d} task={regression_loss.item():.4f} "
            f"balance={bal_loss.item():.4f} load={load.tolist()}"
        )

step=00 task=4.0044 balance=0.0207 load=[0.1979166716337204, 0.1875, 0.21875, 0.3958333432674408]
step=15 task=3.1807 balance=0.0201 load=[0.234375, 0.25, 0.296875, 0.21875]
step=30 task=2.0641 balance=0.0200 load=[0.265625, 0.2864583432674408, 0.1927083283662796, 0.2552083432674408]


step=45 task=1.6677 balance=0.0202 load=[0.3020833432674408, 0.2447916716337204, 0.28125, 0.171875]
step=60 task=1.3196 balance=0.0200 load=[0.2395833283662796, 0.2864583432674408, 0.234375, 0.2395833283662796]


## 8. Capacity、token dropping 与分布式 Expert Parallel

到目前为止，每个专家接收任意数量 token。生产系统通常为每个专家设置容量：

$$
C=\left\lceil \text{capacity\_factor}\cdot\frac{TK}{E}\right\rceil.
$$

- `capacity_factor > 1` 留出不均衡余量；
- 超过容量的 token 可丢弃、走残差/共享专家，或重新路由；
- capacity 太小：丢 token 多，质量下降；太大：padding/显存与通信浪费。

当专家分散在不同 GPU 上时，典型流程是：

```text
本卡 token → Router → 按目标专家排序/打包
          → All-to-All 把 token 发到专家所在 GPU
          → 各 GPU 计算本地专家
          → All-to-All 把输出发回原 GPU
          → inverse permutation + route-weight combine
```

常见并行维度：Data Parallel 复制模型；Tensor Parallel 切分单个专家矩阵；Expert Parallel 把不同专家放到不同设备。通信经常比矩阵乘本身更容易成为瓶颈，因此 token 排序、容量控制、通信/计算重叠和 fused kernel 很关键。

### FLOPs 与墙钟时间要分开看

理论上 routed 部分每 token 只计算 $K/E$ 的专家，但以下写法仍会全算：

```python
for expert in experts:
    expert_out = expert(x_flat)  # 每个专家都处理全部 T 个 token
    output += expert_out * mask
```

它的 FLOPs 接近 Dense MoE。真正稀疏应先用索引取出该专家的 token 子集，或使用专门的稀疏 kernel。

## 9. Shared Expert、Top-1 与 Top-2 怎么选

### Shared Expert

共享专家始终激活，可学习跨领域通用变换；routed experts 更专注于差异化模式。代价是所有 token 都多一次 dense FFN。常见组合是：

$$y=y_{routed}+y_{shared}.$$

### Top-1

- 计算和通信最低；
- 路由更硬，对负载均衡更敏感；
- 每个 token 只得到一个专家视角。

### Top-2

- 两个专家加权组合，通常更平滑、更有容量；
- routed 计算量和通信量大约是 Top-1 的两倍；
- 负载统计要明确按 token 还是按 assignment 计数。

### 常用超参数起点（不是定律）

- `num_experts`: 4～8（小模型先从 4 开始）；
- `top_k`: 1 或 2；
- `aux_loss coefficient`: $10^{-3}$～$10^{-2}$，结合未加权损失量级调；
- `capacity_factor`: 1.0～1.25；
- shared expert：任务共享模式很强或担心 token dropping 时有帮助。

## 10. 对照 `agent_policy` 的真实实现

核心文件：

```text
agent_policy/src/models/components/encoder/moe.py
├── SwiGLUExpertMLP     单个 SwiGLU 专家，H → I → H
├── MoEGate             线性 Router、softmax、Top-k、均衡项
├── _AddAuxiliaryLoss   自定义 autograd：保持 forward 数值不变，注入辅助梯度
└── MOELayer
    ├── experts         E 个 routed experts
    ├── shared_experts  可选 shared expert
    ├── _forward_train  复制 K 份后按 mask 只算专家子集
    └── _forward_infer  每个专家对全部 token 计算，再乘 mask
```

### `MOELayer.forward` 中的形状流

假设 `x [B,N,H]`，令 `T=B*N`：

| 变量 | 形状 | 含义 |
|---|---|---|
| `x_flat` | `[T,H]` | 展平 token |
| `logits`, `scores` | `[T,E]` | 全专家路由 logits/概率 |
| `topk_idx` | `[T,K]` | 每 token 的专家编号 |
| `topk_weight` | `[T,K]` | Top-K 重归一化权重 |
| `x_repeated` | `[T*K,H]` | 每 token 复制 K 份 |
| `y`（合并前） | `[T,K,H]` | K 个专家候选输出 |
| `y`（合并后） | `[T,H]` | 沿 K 加权求和 |
| 最终 `y` | `[B,N,H]` | 恢复原形状并加 shared expert |

### 接入网络的位置

- Encoder：`encoder/encoder.py` 的 `PrefusionBlock.mlp_branch`；
- Decoder：`decoder/decoder.py` 的双流/单流 block MLP 分支；
- 模型：`fmslm.py` 汇总 decoder 返回的 `moe_aux_loss`；
- Task：`tasks/flow_matching_task.py` 把规划、一致性、SigLIP、MoE 项组合成总损失。

因此 MoE 是对 Transformer block 的 MLP/FFN 分支进行替换，Attention 分支仍正常工作；输入与输出 hidden dimension 不变。

## 11. `agent_policy` 配置逐项解释

`agent_policy/configs/model/fmslm.yaml` 中目前的模型级配置为：

```yaml
moe:
  num_experts: 8               # E：routed expert 总数
  num_experts_per_tok: 2       # K：每个 token 激活两个专家
  moe_intermediate_size: 384   # I：每个 routed SwiGLU 专家的中间维
  num_shared_experts: 1        # 共享专家数量/宽度倍率
  aux_loss_alpha: 0.01         # 负载均衡系数
  moe_first_k_dense_replace: 0 # 前多少层保留 Dense FFN
```

Decoder 子配置 `configs/model/decoder/fmslm.yaml` 还包含：

```yaml
dit_config:
  use_moe: true
  moe_config:
    num_experts: 8
    num_experts_per_tok: 2
    moe_intermediate_size: 384
    num_shared_experts: 1
    aux_loss_alpha: 0.01
```

修改配置时要同时确认模型构造链最终读的是哪一层配置，避免“改了模型级参数，但 decoder 仍使用自己的 `dit_config.moe_config`”。`moe_first_k_dense_replace` 只有在构造层 flags 时被实际消费才会生效，不能仅凭 YAML 字段名假定已经生效。

### 参数/计算粗估

当前单个 routed 专家（$H=192,I=384$）权重约 $3HI=221,184$；8 个约 177 万参数。Top-2 理想稀疏 routed 计算相当于每 token 运行 2 个专家，另加始终运行的 shared expert。实际总量还要加 Router、Attention、投影层以及多层重复。

In [7]:
# 用公式核对 agent_policy 默认专家参数规模。
project_H, project_I, project_E, project_K = 192, 384, 8, 2
one_expert_params = 3 * project_H * project_I
routed_params = project_E * one_expert_params
router_params = project_E * project_H
shared_params = one_expert_params
print(f"单个 SwiGLU 专家参数: {one_expert_params:,}")
print(f"8 个 routed experts: {routed_params:,}")
print(f"Router: {router_params:,}")
print(f"1 个 shared expert: {shared_params:,}")
print(f"MoE 层相关权重合计: {routed_params + router_params + shared_params:,}")
print("每 token 激活的专家数: routed", project_K, "+ shared 1")

单个 SwiGLU 专家参数: 221,184
8 个 routed experts: 1,769,472
Router: 1,536
1 个 shared expert: 221,184
MoE 层相关权重合计: 1,992,192
每 token 激活的专家数: routed 2 + shared 1


## 12. 当前项目实现中需要特别留意的两点

这部分描述的是**当前源码行为**，不是 MoE 的普遍规定。

### 12.1 推理路径不是真正的计算稀疏

训练路径 `_forward_train` 先选 token 子集，每个专家只处理自己的 token；但 `_forward_infer` 当前对每个专家都执行：

```python
expert_out = expert(x_flat)  # [T,H]，所有 token
expert_weight = ...          # 非本专家 token 权重为 0
expert_cache += expert_out * expert_weight
```

这在数值语义上能得到正确的 Top-k 加权结果，也更容易做静态图/ONNX 导出，但计算量是所有 $E$ 个专家全算。若目标是 GPU 稀疏加速，需要 token grouping/scatter-gather 或 fused MoE kernel；若目标是部署导出稳定性，当前 dense-mask 路径则有合理工程取舍。

### 12.2 辅助损失目前存在重复施加风险

当前链路同时存在：

1. `MoEGate` 内部已经计算 `aux_loss = raw_balance * aux_loss_alpha`；
2. `_AddAuxiliaryLoss.apply(y, aux_loss)` 将该损失梯度注入 MoE 输出；
3. `FlowMatchingTask` 又显式加入 `self.model.aux_loss_alpha * moe_aux_loss`。

因此显式分支上的系数可能变成 $\alpha^2$，而注入分支又额外贡献一次梯度。更清晰的标准做法通常二选一：

- **推荐用于可读性**：Gate 返回未加权 `raw_balance_loss`，Task 统一执行 `total += alpha * raw_balance_loss`；
- 特殊框架若难以穿透返回 loss，可使用 `_AddAuxiliaryLoss` 注入，但就不要再显式重复相加。

修改项目前应先用梯度回归测试确认原设计意图；本笔记只揭示接口语义，不自动改动项目训练行为。

## 13. 诊断 MoE 是否健康

仅观察总 loss 不够，建议每个 MoE 层记录：

- `expert_load_i`：每个专家实际 assignment 比例 $f_i$；
- `router_prob_i`：平均概率 $P_i$；
- `top1_fraction` / `top2_fraction`；
- `load_cv = std(load) / mean(load)`：越高越不均衡；
- router entropy：过低可能路由过硬/塌缩，过高可能没有专门化；
- dropped-token rate（有 capacity 时）；
- 每个专家参数与梯度范数；
- routed/shared 输出范数，防止某一路完全主导；
- 实际 wall-clock、峰值显存、all-to-all 时间，而不仅是理论 FLOPs。

下面给出可直接复用的无梯度统计函数。

In [8]:
@torch.no_grad()
def router_diagnostics(scores, topk_idx):
    '''scores [T,E]，topk_idx [T,K]；返回 Python 标量/列表，便于日志记录。'''
    E_local = scores.shape[-1]
    assignment = F.one_hot(topk_idx, num_classes=E_local).float()  # [T,K,E]
    load = assignment.mean(dim=(0, 1))                             # [E]
    mean_prob = scores.mean(dim=0)                                 # [E]
    entropy_per_token = -(scores.clamp_min(1e-9) * scores.clamp_min(1e-9).log()).sum(-1)
    return {
        "load": load.cpu().tolist(),
        "mean_probability": mean_prob.cpu().tolist(),
        "load_cv": (load.std(unbiased=False) / load.mean().clamp_min(1e-9)).item(),
        "router_entropy": entropy_per_token.mean().item(),
        "max_entropy": math.log(E_local),
        "unused_experts": int((load == 0).sum()),
    }


diagnostics = router_diagnostics(stats["route"].scores, stats["route"].topk_idx)
for key, value in diagnostics.items():
    print(f"{key}: {value}")

load: [0.30000001192092896, 0.30000001192092896, 0.30000001192092896, 0.10000000149011612]
mean_probability: [0.2768665850162506, 0.25089192390441895, 0.27333903312683105, 0.1989024579524994]
load_cv: 0.34641018509864807
router_entropy: 1.2675950527191162
max_entropy: 1.3862943611198906
unused_experts: 0


## 14. 常见错误清单

1. **Top-k 后不重归一化**：输出幅度随保留概率质量变化；有些论文故意不重归一化，但必须明确。
2. **把 `topk_idx` 当可微变量**：离散索引本身没有梯度；梯度经选中概率和专家输出传播。
3. **训练与推理算法数值不一致**：两条 dispatch 路径应在 `eval/no_grad` 下做误差对齐测试。
4. **对 padding token 也路由**：padding 会污染负载统计和专家训练，应在 Router/统计时排除。
5. **只看均衡 loss，不看每层负载**：全局平均可能掩盖某一层塌缩。
6. **辅助损失权重乘两次**：函数内部与 Task 层职责不清时最常见。
7. **宣称稀疏但所有专家全算**：mask 输出不等于节省专家前向计算。
8. **混合精度下 Router 不稳定**：路由 logits/softmax 常保留 FP32，再把 Top-k 权重转回输入 dtype。
9. **没有容量与通信设计**：单卡小实验能跑，不代表多卡吞吐可扩展。
10. **假定专家会自然形成可解释分工**：专门化是涌现结果，不应仅凭专家编号下结论。

---

## 15. 小结

- MoE 用 Router 将 token 动态分配给少量专家，以较低的激活计算换取更大的参数容量；
- 完整 MoE 不只是 `ModuleList`：还包含路由、dispatch、combine、均衡、capacity 与并行通信；
- `agent_policy` 使用 Top-2、SwiGLU routed experts 与 shared expert，并替换 Encoder/Decoder block 的 MLP 分支；
- 张量主线是 `[B,N,H] → [T,E] → [T,K] → [T,K,H] → [B,N,H]`；
- 工程评估必须同时看模型质量、专家负载、实际 FLOPs、吞吐、显存和通信。

建议练习：加入 padding mask；实现 capacity 与 token dropping；实现一个按专家排序的推理 dispatch；比较 Top-1/Top-2 的质量与吞吐；为 `agent_policy` 添加逐层 router diagnostics。